# 24. Async objects, concurrency, and thread safety

**Level:** Advanced

**Learning outcome:** Use awaitable methods, bound in-flight operations, own task lifetimes, and inspect every threaded result.

**Prerequisites:** 04, 13, and 15: dependencies, cleanup, and error handling.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](23_New_Initialization_and_Subclass_Hooks.ipynb) | [Next notebook](25_Telemetry_Counters_Windows_and_Alarms.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Bound concurrent work, preserve errors, and avoid blocking the notebook event loop.

**Concurrency** means tasks can make progress over overlapping periods. It is not the same as
CPU parallelism. asyncio is cooperative: a coroutine gives other tasks opportunities to run
at awaits. A normal blocking transport call inside async def still blocks the event loop.
Use an asynchronous adapter or move appropriate blocking I/O to a bounded thread pool.

**Notebook event loops.** Jupyter usually already has an event loop. These cells use top-level
await rather than asyncio.run inside the notebook. In a normal script, put equivalent work
in async main() and call asyncio.run(main()) once at the entry point. Each async cell defines
all its classes and imports itself.

**Structured concurrency.** A TaskGroup owns a collection of tasks; exiting its context waits
for them and handles task failures according to its contract. Do not create detached tasks
and forget their results. Cancellation is a control signal; cleanup belongs in finally or an
async context manager, and CancelledError should normally propagate after cleanup.

**Bound work.** A semaphore limits simultaneous operations, not total queue size. A large
inventory may also need a bounded producer/consumer queue. Timeouts and rate limits address
different problems. Avoid launching a task for every device in a very large fleet at once.

**Shared state.** Threads need synchronization around compound updates. A Lock can protect an
invariant, but keep critical sections short and do not hold a regular blocking lock across
an await. ThreadPoolExecutor is suitable for bounded blocking I/O; ordinary CPU-bound Python
work and interpreter build details require separate performance reasoning. These lessons
use tiny local workloads and no real network, real sleeps, or long-running background tasks.

**Pause and explain:** Explain top-level await, why blocking I/O still blocks inside async def, and what a semaphore limits.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 081: Async method gathers offline observations

**Core concept:** An async object API can coordinate awaitable operations.

**Network-engineering purpose:** A poller can collect several simulated device outcomes while preserving result order.

### Read the implementation in this order
1. Define async read on the poller.
2. Yield with asyncio.sleep(0) without a real delay.
3. Gather two coroutine results.
4. Assert output order follows input order.

**Common mistake / boundary:** async def alone does not make blocking libraries nonblocking.

**Practice after the ten worked scenarios:** Replace gather with explicit tasks and inspect each task's result after completion.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0801 - Async method gathers offline observations: branch scenario

**Scenario 01/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['branch-edge-01:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [1]:
# Example 0801: Async method gathers offline observations - branch
case = {'hostname': 'branch-edge-01'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['branch-edge-01:up', 'lab-spare:up']


### Example 0802 - Async method gathers offline observations: campus scenario

**Scenario 02/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['campus-access-02:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [2]:
# Example 0802: Async method gathers offline observations - campus
case = {'hostname': 'campus-access-02'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['campus-access-02:up', 'lab-spare:up']


### Example 0803 - Async method gathers offline observations: dc scenario

**Scenario 03/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['dc-leaf-03:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [3]:
# Example 0803: Async method gathers offline observations - dc
case = {'hostname': 'dc-leaf-03'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['dc-leaf-03:up', 'lab-spare:up']


### Example 0804 - Async method gathers offline observations: wan scenario

**Scenario 04/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wan-border-04:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [4]:
# Example 0804: Async method gathers offline observations - wan
case = {'hostname': 'wan-border-04'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['wan-border-04:up', 'lab-spare:up']


### Example 0805 - Async method gathers offline observations: lab scenario

**Scenario 05/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['lab-router-05:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [5]:
# Example 0805: Async method gathers offline observations - lab
case = {'hostname': 'lab-router-05'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['lab-router-05:up', 'lab-spare:up']


### Example 0806 - Async method gathers offline observations: wireless scenario

**Scenario 06/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['wireless-dist-06:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [6]:
# Example 0806: Async method gathers offline observations - wireless
case = {'hostname': 'wireless-dist-06'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['wireless-dist-06:up', 'lab-spare:up']


### Example 0807 - Async method gathers offline observations: remote scenario

**Scenario 07/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['remote-gateway-07:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [7]:
# Example 0807: Async method gathers offline observations - remote
case = {'hostname': 'remote-gateway-07'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['remote-gateway-07:up', 'lab-spare:up']


### Example 0808 - Async method gathers offline observations: core scenario

**Scenario 08/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['core-router-08:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [8]:
# Example 0808: Async method gathers offline observations - core
case = {'hostname': 'core-router-08'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['core-router-08:up', 'lab-spare:up']


### Example 0809 - Async method gathers offline observations: mgmt scenario

**Scenario 09/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['mgmt-switch-09:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [9]:
# Example 0809: Async method gathers offline observations - mgmt
case = {'hostname': 'mgmt-switch-09'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['mgmt-switch-09:up', 'lab-spare:up']


### Example 0810 - Async method gathers offline observations: backup scenario

**Scenario 10/10; pattern 081; Advanced.** This is a self-contained offline exercise. A poller can collect several simulated device outcomes while preserving result order.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define async read on the poller. Assert output order follows input order. Follow the class-specific steps under Pattern 081 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
['backup-edge-10:up', 'lab-spare:up']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** async def alone does not make blocking libraries nonblocking.

In [10]:
# Example 0810: Async method gathers offline observations - backup
case = {'hostname': 'backup-edge-10'}

import asyncio

class Poller:
    async def read(self, hostname):
        await asyncio.sleep(0)
        return hostname + ":up"

poller = Poller()
names = [case["hostname"], "lab-spare"]
results = await asyncio.gather(*(poller.read(name) for name in names))
assert results == [name + ":up" for name in names]
print(results)


['backup-edge-10:up', 'lab-spare:up']


## Pattern 082: Semaphore bounds in-flight operations

**Core concept:** A semaphore guards a concurrent section around each operation.

**Network-engineering purpose:** A simulated poller should never exceed its configured in-flight budget.

### Read the implementation in this order
1. Create one shared semaphore.
2. Track active and peak counts inside its async context.
3. Use finally to decrement active.
4. Gather a small workload and assert the measured peak is within the limit.

**Common mistake / boundary:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

**Practice after the ten worked scenarios:** Use a bounded queue and a fixed number of workers for a much larger input.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0811 - Semaphore bounds in-flight operations: branch scenario

**Scenario 01/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [11]:
# Example 0811: Semaphore bounds in-flight operations - branch
case = {'hostname': 'branch-edge-01', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


### Example 0812 - Semaphore bounds in-flight operations: campus scenario

**Scenario 02/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [12]:
# Example 0812: Semaphore bounds in-flight operations - campus
case = {'hostname': 'campus-access-02', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 2 limit: 2


### Example 0813 - Semaphore bounds in-flight operations: dc scenario

**Scenario 03/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [13]:
# Example 0813: Semaphore bounds in-flight operations - dc
case = {'hostname': 'dc-leaf-03', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 3 limit: 3


### Example 0814 - Semaphore bounds in-flight operations: wan scenario

**Scenario 04/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [14]:
# Example 0814: Semaphore bounds in-flight operations - wan
case = {'hostname': 'wan-border-04', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


### Example 0815 - Semaphore bounds in-flight operations: lab scenario

**Scenario 05/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [15]:
# Example 0815: Semaphore bounds in-flight operations - lab
case = {'hostname': 'lab-router-05', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 2 limit: 2


### Example 0816 - Semaphore bounds in-flight operations: wireless scenario

**Scenario 06/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [16]:
# Example 0816: Semaphore bounds in-flight operations - wireless
case = {'hostname': 'wireless-dist-06', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 3 limit: 3


### Example 0817 - Semaphore bounds in-flight operations: remote scenario

**Scenario 07/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [17]:
# Example 0817: Semaphore bounds in-flight operations - remote
case = {'hostname': 'remote-gateway-07', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


### Example 0818 - Semaphore bounds in-flight operations: core scenario

**Scenario 08/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |
| `concurrency` | `2` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 2 limit: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [18]:
# Example 0818: Semaphore bounds in-flight operations - core
case = {'hostname': 'core-router-08', 'concurrency': 2}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 2 limit: 2


### Example 0819 - Semaphore bounds in-flight operations: mgmt scenario

**Scenario 09/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |
| `concurrency` | `3` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 3 limit: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [19]:
# Example 0819: Semaphore bounds in-flight operations - mgmt
case = {'hostname': 'mgmt-switch-09', 'concurrency': 3}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 3 limit: 3


### Example 0820 - Semaphore bounds in-flight operations: backup scenario

**Scenario 10/10; pattern 082; Advanced.** This is a self-contained offline exercise. A simulated poller should never exceed its configured in-flight budget.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |
| `concurrency` | `1` |

**Logic to trace:** Create one shared semaphore. Gather a small workload and assert the measured peak is within the limit. Follow the class-specific steps under Pattern 082 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
peak: 1 limit: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A semaphore alone does not bound memory used by a huge list of scheduled tasks.

In [20]:
# Example 0820: Semaphore bounds in-flight operations - backup
case = {'hostname': 'backup-edge-10', 'concurrency': 1}

import asyncio

class BoundedPoller:
    def __init__(self, limit):
        self.semaphore = asyncio.Semaphore(limit)
        self.active, self.peak = 0, 0

    async def read(self, hostname):
        async with self.semaphore:
            self.active += 1
            self.peak = max(self.peak, self.active)
            try:
                await asyncio.sleep(0)
                return hostname
            finally:
                self.active -= 1

limit = case["concurrency"]
poller = BoundedPoller(limit)
names = [case["hostname"] + f"-{index}" for index in range(6)]
result = await asyncio.gather(*(poller.read(name) for name in names))
assert result == names
assert poller.peak == min(limit, len(names)) and poller.active == 0
print("peak:", poller.peak, "limit:", limit)


peak: 1 limit: 1


## Pattern 083: TaskGroup owns concurrent tasks

**Core concept:** A structured task scope keeps task lifetime tied to the calling operation.

**Network-engineering purpose:** A small fleet collection waits for every child before building its final report.

### Read the implementation in this order
1. Define an async collector method.
2. Create tasks inside TaskGroup.
3. Read results only after the group exits.
4. Check the report contains all expected values.

**Common mistake / boundary:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

**Practice after the ten worked scenarios:** Make one task raise a domain error and inspect the resulting ExceptionGroup in a separate exercise.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0821 - TaskGroup owns concurrent tasks: branch scenario

**Scenario 01/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'branch-edge-01', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [21]:
# Example 0821: TaskGroup owns concurrent tasks - branch
case = {'hostname': 'branch-edge-01'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'branch-edge-01', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0822 - TaskGroup owns concurrent tasks: campus scenario

**Scenario 02/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'campus-access-02', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [22]:
# Example 0822: TaskGroup owns concurrent tasks - campus
case = {'hostname': 'campus-access-02'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'campus-access-02', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0823 - TaskGroup owns concurrent tasks: dc scenario

**Scenario 03/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'dc-leaf-03', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [23]:
# Example 0823: TaskGroup owns concurrent tasks - dc
case = {'hostname': 'dc-leaf-03'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'dc-leaf-03', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0824 - TaskGroup owns concurrent tasks: wan scenario

**Scenario 04/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'wan-border-04', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [24]:
# Example 0824: TaskGroup owns concurrent tasks - wan
case = {'hostname': 'wan-border-04'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'wan-border-04', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0825 - TaskGroup owns concurrent tasks: lab scenario

**Scenario 05/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'lab-router-05', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [25]:
# Example 0825: TaskGroup owns concurrent tasks - lab
case = {'hostname': 'lab-router-05'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'lab-router-05', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0826 - TaskGroup owns concurrent tasks: wireless scenario

**Scenario 06/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'wireless-dist-06', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [26]:
# Example 0826: TaskGroup owns concurrent tasks - wireless
case = {'hostname': 'wireless-dist-06'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'wireless-dist-06', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0827 - TaskGroup owns concurrent tasks: remote scenario

**Scenario 07/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'remote-gateway-07', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [27]:
# Example 0827: TaskGroup owns concurrent tasks - remote
case = {'hostname': 'remote-gateway-07'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'remote-gateway-07', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0828 - TaskGroup owns concurrent tasks: core scenario

**Scenario 08/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'core-router-08', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [28]:
# Example 0828: TaskGroup owns concurrent tasks - core
case = {'hostname': 'core-router-08'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'core-router-08', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0829 - TaskGroup owns concurrent tasks: mgmt scenario

**Scenario 09/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'mgmt-switch-09', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [29]:
# Example 0829: TaskGroup owns concurrent tasks - mgmt
case = {'hostname': 'mgmt-switch-09'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'mgmt-switch-09', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


### Example 0830 - TaskGroup owns concurrent tasks: backup scenario

**Scenario 10/10; pattern 083; Advanced.** This is a self-contained offline exercise. A small fleet collection waits for every child before building its final report.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Define an async collector method. Check the report contains all expected values. Follow the class-specific steps under Pattern 083 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{'hostname': 'backup-edge-10', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Production callers must handle grouped failures deliberately rather than assume all tasks succeed.

In [30]:
# Example 0830: TaskGroup owns concurrent tasks - backup
case = {'hostname': 'backup-edge-10'}

import asyncio

class Collector:
    async def read(self, name):
        await asyncio.sleep(0)
        return {"hostname": name, "healthy": True}

collector = Collector()
names = [case["hostname"], "lab-secondary"]
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(collector.read(name)) for name in names]
results = [task.result() for task in tasks]
assert [item["hostname"] for item in results] == names
assert all(item["healthy"] for item in results)
print(results)


[{'hostname': 'backup-edge-10', 'healthy': True}, {'hostname': 'lab-secondary', 'healthy': True}]


## Pattern 084: Lock protects a shared threaded counter

**Core concept:** A lock protects the complete read-modify-write operation.

**Network-engineering purpose:** A metrics collector should not rely on incidental interpreter behavior for compound updates.

### Read the implementation in this order
1. Store a Lock on the counter.
2. Increment while holding it.
3. Run a bounded batch through a context-managed executor.
4. Assert the final total equals the submitted work.

**Common mistake / boundary:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

**Practice after the ten worked scenarios:** Track successes and failures under one lock while preserving a total-attempt invariant.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0831 - Lock protects a shared threaded counter: branch scenario

**Scenario 01/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `6` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 6
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [31]:
# Example 0831: Lock protects a shared threaded counter - branch
case = {'jobs': 6}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 6


### Example 0832 - Lock protects a shared threaded counter: campus scenario

**Scenario 02/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `7` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 7
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [32]:
# Example 0832: Lock protects a shared threaded counter - campus
case = {'jobs': 7}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 7


### Example 0833 - Lock protects a shared threaded counter: dc scenario

**Scenario 03/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `8` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 8
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [33]:
# Example 0833: Lock protects a shared threaded counter - dc
case = {'jobs': 8}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 8


### Example 0834 - Lock protects a shared threaded counter: wan scenario

**Scenario 04/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `9` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 9
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [34]:
# Example 0834: Lock protects a shared threaded counter - wan
case = {'jobs': 9}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 9


### Example 0835 - Lock protects a shared threaded counter: lab scenario

**Scenario 05/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `10` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 10
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [35]:
# Example 0835: Lock protects a shared threaded counter - lab
case = {'jobs': 10}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 10


### Example 0836 - Lock protects a shared threaded counter: wireless scenario

**Scenario 06/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `11` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 11
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [36]:
# Example 0836: Lock protects a shared threaded counter - wireless
case = {'jobs': 11}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 11


### Example 0837 - Lock protects a shared threaded counter: remote scenario

**Scenario 07/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `12` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 12
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [37]:
# Example 0837: Lock protects a shared threaded counter - remote
case = {'jobs': 12}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 12


### Example 0838 - Lock protects a shared threaded counter: core scenario

**Scenario 08/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `13` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 13
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [38]:
# Example 0838: Lock protects a shared threaded counter - core
case = {'jobs': 13}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed:

 13


### Example 0839 - Lock protects a shared threaded counter: mgmt scenario

**Scenario 09/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `14` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 14
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [39]:
# Example 0839: Lock protects a shared threaded counter - mgmt
case = {'jobs': 14}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 14


### Example 0840 - Lock protects a shared threaded counter: backup scenario

**Scenario 10/10; pattern 084; Advanced.** This is a self-contained offline exercise. A metrics collector should not rely on incidental interpreter behavior for compound updates.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `jobs` | `15` |

**Logic to trace:** Store a Lock on the counter. Assert the final total equals the submitted work. Follow the class-specific steps under Pattern 084 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
completed: 15
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not assume the GIL makes all compound operations safe across Python implementations or builds.

In [40]:
# Example 0840: Lock protects a shared threaded counter - backup
case = {'jobs': 15}

from concurrent.futures import ThreadPoolExecutor
from threading import Lock

class Counter:
    def __init__(self):
        self.value, self.lock = 0, Lock()

    def increment(self, unused):
        with self.lock:
            self.value += 1

counter = Counter()
count = case["jobs"]
with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(counter.increment, range(count)))
assert counter.value == count
print("completed:", counter.value)


completed: 15


## Pattern 085: Bounded executor preserves per-item failures

**Core concept:** Future results must be inspected so worker errors reach the caller.

**Network-engineering purpose:** A fleet report can distinguish one failed simulated read from successful devices.

### Read the implementation in this order
1. Submit known targets to a small pool.
2. Associate each future with its target.
3. Read every result and catch only the expected TimeoutError.
4. Check success and failure counts and the failed hostname.

**Common mistake / boundary:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

**Practice after the ten worked scenarios:** Return a structured result dataclass instead of separate success and failure lists.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0841 - Bounded executor preserves per-item failures: branch scenario

**Scenario 01/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['branch-edge-01:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [41]:
# Example 0841: Bounded executor preserves per-item failures - branch
case = {'hostname': 'branch-edge-01'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['branch-edge-01:up'] failure: ['lab-failed']


### Example 0842 - Bounded executor preserves per-item failures: campus scenario

**Scenario 02/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['campus-access-02:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [42]:
# Example 0842: Bounded executor preserves per-item failures - campus
case = {'hostname': 'campus-access-02'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['campus-access-02:up'] failure: ['lab-failed']


### Example 0843 - Bounded executor preserves per-item failures: dc scenario

**Scenario 03/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['dc-leaf-03:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [43]:
# Example 0843: Bounded executor preserves per-item failures - dc
case = {'hostname': 'dc-leaf-03'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['dc-leaf-03:up'] failure: ['lab-failed']


### Example 0844 - Bounded executor preserves per-item failures: wan scenario

**Scenario 04/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['wan-border-04:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [44]:
# Example 0844: Bounded executor preserves per-item failures - wan
case = {'hostname': 'wan-border-04'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success:

 ['wan-border-04:up'] failure: ['lab-failed']


### Example 0845 - Bounded executor preserves per-item failures: lab scenario

**Scenario 05/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['lab-router-05:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [45]:
# Example 0845: Bounded executor preserves per-item failures - lab
case = {'hostname': 'lab-router-05'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success:

 ['lab-router-05:up'] failure: ['lab-failed']


### Example 0846 - Bounded executor preserves per-item failures: wireless scenario

**Scenario 06/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['wireless-dist-06:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [46]:
# Example 0846: Bounded executor preserves per-item failures - wireless
case = {'hostname': 'wireless-dist-06'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['wireless-dist-06:up'] failure: ['lab-failed']


### Example 0847 - Bounded executor preserves per-item failures: remote scenario

**Scenario 07/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['remote-gateway-07:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [47]:
# Example 0847: Bounded executor preserves per-item failures - remote
case = {'hostname': 'remote-gateway-07'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['remote-gateway-07:up'] failure: ['lab-failed']


### Example 0848 - Bounded executor preserves per-item failures: core scenario

**Scenario 08/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['core-router-08:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [48]:
# Example 0848: Bounded executor preserves per-item failures - core
case = {'hostname': 'core-router-08'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['core-router-08:up'] failure: ['lab-failed']


### Example 0849 - Bounded executor preserves per-item failures: mgmt scenario

**Scenario 09/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['mgmt-switch-09:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [49]:
# Example 0849: Bounded executor preserves per-item failures - mgmt
case = {'hostname': 'mgmt-switch-09'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['mgmt-switch-09:up'] failure: ['lab-failed']


### Example 0850 - Bounded executor preserves per-item failures: backup scenario

**Scenario 10/10; pattern 085; Advanced.** This is a self-contained offline exercise. A fleet report can distinguish one failed simulated read from successful devices.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Submit known targets to a small pool. Check success and failure counts and the failed hostname. Follow the class-specific steps under Pattern 085 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
success: ['backup-edge-10:up'] failure: ['lab-failed']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Fire-and-forget futures can hide failures; inspecting every result is part of completion.

In [50]:
# Example 0850: Bounded executor preserves per-item failures - backup
case = {'hostname': 'backup-edge-10'}

from concurrent.futures import ThreadPoolExecutor

class Poller:
    def read(self, hostname):
        if hostname == "lab-failed":
            raise TimeoutError("simulated read timeout")
        return hostname + ":up"

successes, failures = [], []
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(Poller().read, name): name
               for name in [case["hostname"], "lab-failed"]}
    for future, name in futures.items():
        try:
            successes.append(future.result())
        except TimeoutError:
            failures.append(name)
assert successes == [case["hostname"] + ":up"]
assert failures == ["lab-failed"]
print("success:", successes, "failure:", failures)


success: ['backup-edge-10:up'] failure: ['lab-failed']


## Check your understanding: Async objects, concurrency, and thread safety

**You should now be able to:** Use awaitable methods, bound in-flight operations, own task lifetimes, and inspect every threaded result.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.